In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install conda-forge::r-cluster conda-forge::r-remotes conda-forge::r-devtools conda-forge::r-fnn")
system("conda install conda-forge::r-factominer conda-forge::r-caret conda-forge::r-factoextra conda-forge::r-rlang")

In [ ]:
install.packages("remotes")
remotes::install_github("bioRgeo/bioregion")

# install.packages("devtools")
# devtools::install_github("bioRgeo/bioregion")

In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse) # because who can live without the tidyverse?
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)

library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

library(data.table)
library(FNN)
library(cluster)
library(bioregion)

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.filename <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
    } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename){
    filename.length <- nchar(filename)

    start <- 1
    end <- filename.length-3
    unzip.filename <- substr(filename,start, end)

    
    if(!file.exists(unzip.filename)){        
        R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}

In [ ]:
# ==============================================================================
# FETCH DATASETS 
# ==============================================================================
bathy_table <- paste(outputDir,"netcdf_depth.csv", sep="/")
sst_table <- paste(outputDir, "raw_sst_table.tsv", sep="/")
iceC_table <- paste(outputDir,"Ice_CopernicusMarineData.tsv", sep="/")

bathy_df <- table.read(bathy_table, header = T)
sst_df <- table.read(sst_table, header = T)
icsC_df <- table.read(iceC_table, header = T)


# Concatenate the Dataframes

In [ ]:
# ----------------
# Filter Out the outerbound values

## 1. Depth
Antarctic_depth_coords <- bathy_df %>%
  filter(
    lon >= 30, lon <= 150,
    lat >= -70, lat <= -60
  )
head(Antarctic_depth_coords)

## 2. SST
Antarctic_sst_coords <- sst_df %>%
  filter(
    lon >= 30, lon <= 150,
    lat >= -70, lat <= -60
  )
head(Antarctic_sst_coords)

## 3. IceConc (Ice Concentration)
Antarctic_iceC_coords <- as.data.frame(icsC_df) %>%
  dplyr::mutate(
    month = format(time, "%m") # Apply month format
  ) %>%
  dplyr::filter(
    lon >= 30, lon <= 150, lat >= -70, lat <= -60,
    (month %in% c("01", "02", "03", "12")) | (month == "04" & format(time, "%d") <= "01")
  ) %>%
    dplyr::group_by(lon, lat) %>% # Groups ONLY BY COORDINATES
    dplyr::summarise(
    ice_mean = mean(iceC, na.rm = TRUE), # Compute the average Ice concentration in summer
    .groups = "drop"
  )
head(Antarctic_iceC_coords)

In [ ]:
# ----------------
# Interpolation
## 1. Create a new grid
new.grid.lon <- seq(from=30.0, to=150.0, by=0.05)
new.grid.lat <- seq(from=-70.0, to=-60.0, by=0.05)
new.grid <- expand.grid(lon=new.grid.lon, lat=new.grid.lat)
head(new.grid)

## 2. Add Bathymetry (depth) + Sea Surface Temperature (sst) + IceConc (iceC)
### +  Fill the added columns  
new.grid$sst <- NA
new.grid$depth <- NA
new.grid$iceC <- NA

head(new.grid) ; dim(new.grid)

In [ ]:
grid   <- as.data.table(new.grid)
sst_dt <- as.data.table(Antarctic_sst_coords)
depth_dt <- as.data.table(Antarctic_depth_coords)
ice_dt <- as.data.table(Antarctic_iceC_coords)

In [ ]:
## SST
# Find nearest neighbor for each grid point
nn <- get.knnx(
  data  = as.matrix(sst_dt[, .(lon, lat)]),
  query = as.matrix(grid[, .(lon, lat)]),
  k = 1
)

# Assign interpolated values
grid[, sst := sst_dt$sst[nn$nn.index]]

In [ ]:
## DEPTH
# Find nearest neighbor for each grid point
nn <- get.knnx(
  data  = as.matrix(depth_dt[, .(lon, lat)]),
  query = as.matrix(grid[, .(lon, lat)]),
  k = 1
)

# Assign interpolated values
grid[, depth := depth_dt$depth[nn$nn.index]]

In [ ]:
### ICE CONCENTRATION
# Find nearest neighbor for each grid point
nn <- get.knnx(  data  = as.matrix(ice_dt[, .(lon, lat)]),  query = as.matrix(grid[, .(lon, lat)]),  k = 1)

# Assign interpolated values
grid[, iceC := ice_dt$ice_mean[nn$nn.index]]
grid %>% rename("ice_mean" = "iceC")

# Statistical Analyses

In [ ]:
# Default value of dissimilarity matrix
# clean SST flag
grid[sst == -32767, sst := NA]

X <- grid[, .(sst, depth, iceC)]

# compute Gower distances (small subset!)
distance.matrix <- cluster::daisy(X[1:1000,], metric = "gower")
gower_matrix <- as.matrix(distance.matrix)

In [ ]:
clara.object <- bioregion::nhclu_clara(
    dissimilarity = distance.matrix, n_clust = seq(from = 2, to = dim(distance.matrix)[1] , by= 1))


In [ ]:
# 1. Clean data
df_clean <- grid %>% mutate(sst = ifelse(sst == -32767, NA, sst))

# 2. Choose k
k_values <- 2:10
sil_width <- sapply(k_values, function(k) {
    nhclu_clara(df_clean, n_clust )$silinfo$avg.width
})

best_k <- k_values[which.max(sil_width)]

# 3. Final model
final_model <- clara(df_clean, k = best_k)
df_clean$cluster <- final_model$clustering

In [ ]:
sil_width
k_values
best_k

In [ ]:
# Compute Gower distance once
gower_dist <- daisy(df_clean, metric = "gower")

# Try different k values
k_values <- 2:10
sil_width <- numeric(length(k_values))

for (i in seq_along(k_values)) {
  k <- k_values[i]
  clara_fit <- clara(df_clean, k = k)
  sil_width[i] <- clara_fit$silinfo$avg.width
}

# Plot
plot(k_values, sil_width, type = "b",
     xlab = "Number of clusters (k)",
     ylab = "Average silhouette width")

# Best k
best_k <- k_values[which.max(sil_width)]

In [ ]:
fviz_nbclust(df_clean, clara, method = "gap_stat", k.max = 10)

In [ ]:
gower_dist <- daisy(df_clean,
                    metric = "gower",
                    weights = c(lon = 1, lat = 1, sst = 2, depth = 2, iceC = 2))